In [8]:
wish_list = {
    "Berliner Tageblatt und Handelszeitung": 30913, 
    "Norddeutsche allgemeine Zeitung": 17924, 
    "National-Zeitung": 5142,
    # clarify the "Allgemeine Zeitung"
    "Vorwärts(Berlin)": 17600,
    "Vorwärts(Leipzig)": 317,
    "General-Anzeiger für Dortmund und die Provinz Westfalen, größte und verbreitetste Tageszeitung Westdeutschlands": 10789,
    # clarify "Neueste Mitteilungen"
 }
# total: 82,685

In [18]:
import requests

import urllib.parse

api_key = "OYSi9Dygc0XZ0Nvq2vgPxe4oXNmomCtWWZHM7CVd3Fo7iC0qKge1748029090188"  
headers = {
    "Authorization": f'OAuth oauth_consumer_key="{api_key}"',
    "Accept": "application/json",
}

zdb = "3000142-0"

params = {
    "q": "*:*",
    "fq": [
        "publication_date:[1869-12-31T23:59:59.999Z TO 1940-12-31T23:59:59.999Z]",
        f"zdb_id:{zdb}"
    ],
    "rows": 100,
    "start": 0
}
url = "https://api.deutsche-digitale-bibliothek.de/search/index/newspaper-issues/select"

In [ ]:
response = requests.get(
    url,
    headers=headers,
    params=params
)
print(response.status_code) 

200


In [20]:
response_json = response.json()
num_results = response_json['response']['numFound']
print(f"Number of matching newspaper issues: {num_results}")

Number of matching newspaper issues: 106461


In [13]:
# **** |id|date|page number|chunk|response| ****
import csv
import os

class DatasetCollector:
    def __init__(self, csv_file):
        self.csv_file = csv_file


    def add_row(self, row_dict):
        # Create the CSV file and write the header if it doesn't exist
        if not os.path.exists(self.csv_file):
            with open(self.csv_file, mode='w', encoding='utf-8', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["item_id", "publisher", "title", "pub_date"])
        with open(self.csv_file, mode='a', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=row_dict.keys())
            writer.writerow(row_dict)

    def add_ids_only(self, id_list):
        """
        Appends a list of IDs into a CSV file, one per row in a single column.
        If the file doesn't exist, it creates it with a header.
        """
        file_exists = os.path.isfile(self.csv_file)

        try:
            with open(self.csv_file, mode="a", newline="", encoding="utf-8") as csvfile:
                writer = csv.writer(csvfile)

                # Write header if file is new
                if not file_exists:
                    writer.writerow(["id"])

                # Write each ID
                for item_id in id_list:
                    writer.writerow([item_id])

            print(f"✔ Added {len(id_list)} IDs to {self.csv_file}")
        except Exception as e:
            print(f"❌ Error writing IDs to {self.csv_file}: {e}")

In [14]:
import requests
import time

class DDBAPI:
    def __init__(self, zdb, rows, start):
        self.api_key = "OYSi9Dygc0XZ0Nvq2vgPxe4oXNmomCtWWZHM7CVd3Fo7iC0qKge1748029090188"  
        self.headers = {
            "Authorization": f'OAuth oauth_consumer_key="{self.api_key}"',
            "Accept": "application/json",
        }
        self.zdb = zdb
        self.params = {
            "q": "*:*",
            "fq": [
                "publication_date:[1869-12-31T23:59:59.999Z TO 1940-12-31T23:59:59.999Z]",
                f"zdb_id:{self.zdb}"
            ],
            "rows": rows,
            "start": start
        }
        self.url = "https://api.deutsche-digitale-bibliothek.de/search/index/newspaper-issues/select"

    def set_paging(self, start, rows):
        """Update pagination parameters for the API request."""
        self.params["start"] = start
        self.params["rows"] = rows

    def get_ddb_data(self):
        """
        Fetches newspaper issue IDs from the Deutsche Digitale Bibliothek API.
        Returns a list of item IDs.
        """
        response = requests.get(
            self.url,
            headers=self.headers,
            params=self.params
        )

        return response

    def get_ids(self, response):
        all_data = response.json()
        ids = [doc['id'] for doc in all_data['response']['docs']]
        return ids
    
    def get_meta_data(self, item_id):
        item_url = f"https://api.deutsche-digitale-bibliothek.de/items/{item_id}"
        try:
            response = requests.get(item_url, headers=self.headers, timeout=10)
            response.raise_for_status()  # raises an HTTPError for bad responses (4xx, 5xx)
        except (requests.exceptions.RequestException, Exception) as e:
            print(f"⚠️ Error fetching {item_id}: {e}")
            time.sleep(1)
            return '', '', ''
            
        if response.status_code != 200:
            print(f"❌ Failed to fetch item {item_id}")
            return '', '', ''

        try:
            data = response.json()
            issued = data['edm']['RDF']['ProvidedCHO']['issued']
            publisher = data['edm']['RDF']['ProvidedCHO']['publisher']['$']
            title = data['edm']['RDF']['ProvidedCHO']['title']['$']
            return issued, publisher, title

        except Exception as e:
            print(f"❗ Error with item {item_id}: {e}")
            return '', '', ''

In [21]:
# zdb = "2764651-8"  # Berliner Tageblatt und Handelszeitung
# zdb = "2802868-5"  # Norddeutsche allgemeine Zeitung
# zdb = "2814128-3"  # Vorwärts(Berlin)
zdb = "3000142-0"  # General-Anzeiger für Dortmund und die Provinz Westfalen, größte und verbreitetste Tageszeitung Westdeutschlands

rows = 1000        # number of results per request
start = 0      # starting offset
total = 106461     # total number of IDs to fetch

In [22]:

csv_file = f"ids_only_{zdb}.csv"
collector = DatasetCollector(csv_file)

ddb = DDBAPI(zdb, rows, start)

while start < total:
    print(f"\n➡️ Fetching IDs {start} to {start + rows} ...")

    response = ddb.get_ddb_data()
    if response.status_code != 200:
        print(f"❌ Request failed at start={start}, status={response.status_code}")
        break

    item_ids = ddb.get_ids(response)

    if not item_ids:
        print("⚠️ No more IDs returned, stopping.")
        break

    collector.add_ids_only(item_ids)

    start += rows
    # Update start in params
    ddb.set_paging(start, rows)
    time.sleep(1)  # be nice to the API


➡️ Fetching IDs 0 to 1000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 1000 to 2000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 2000 to 3000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 3000 to 4000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 4000 to 5000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 5000 to 6000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 6000 to 7000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 7000 to 8000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 8000 to 9000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 9000 to 10000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 10000 to 11000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 11000 to 12000 ...
✔ Added 1000 IDs to ids_only_3000142-0.csv

➡️ Fetching IDs 12000 to 13000 ...
✔ Added 1000 IDs to ids_only_3000142-0

In [1]:
import pandas as pd
import glob


In [2]:
def load_and_combine_csvs(folder_path, output_path=None):
    """
    Load all CSV files from a folder and combine them into one DataFrame.
    Optionally save the combined dataset to output_path.
    """
    # Find all CSV files in the folder
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    if not all_files:
        print("No CSV files found in folder.")
        return pd.DataFrame()
    
    # Read and concatenate
    df_list = []
    for file in all_files:
        try:
            df = pd.read_csv(file)
            df_list.append(df)
        except Exception as e:
            print(f"Skipping {file} due to error: {e}")
    
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Optionally save
    if output_path:
        combined_df.to_csv(output_path, index=False)
        print(f"Combined CSV saved to {output_path}")
    
    return combined_df

In [11]:
df = load_and_combine_csvs("./NordDeuAllgem", "./NordDeuAllgem/combined_ids.csv")
print(f"Total combined rows: {len(df)}")

Combined CSV saved to ./NordDeuAllgem/combined_ids.csv
Total combined rows: 145816


In [6]:
def drop_ddb_fulltext(df: pd.DataFrame, id_col: str = "id") -> pd.DataFrame:
    """
    Remove all rows where the given id column contains 'DDB_FULLTEXT'
    and drop duplicate rows.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    id_col : str, default "id"
        Column name that contains IDs.

    Returns
    -------
    pd.DataFrame
        Cleaned dataframe with rows removed and duplicates dropped.
    """
    cleaned = df[~df[id_col].str.contains("DDB_FULLTEXT", na=False)]
    cleaned = cleaned.drop_duplicates()
    return cleaned

In [17]:
df = pd.read_csv("./GAfDortmund/ids_only_3000142-0.csv")
print(len(df))
df = drop_ddb_fulltext(df, "id")
df.to_csv("./GAfDortmund/cleaned_ids.csv", index=False)
print(f"Rows after dropping 'DDB_FULLTEXT': {len(df)}")


106461
Rows after dropping 'DDB_FULLTEXT': 7615


In [11]:
def check_duplicates(file_name):
    df = pd.read_csv(file_name)

    # Find duplicate IDs
    duplicates = df[df.duplicated("id", keep=False)]

    if duplicates.empty:
        print(f"✅ {os.path.basename(file_name)}: No duplicates found.")
    else:
        print(f"⚠️ {os.path.basename(file_name)}: Found {duplicates.shape[0]} duplicate rows:")
        print(duplicates)

    return duplicates



In [18]:
check_duplicates("./GAfDortmund/cleaned_ids.csv")

✅ cleaned_ids.csv: No duplicates found.


,id
